# GOES-R series (GOES-16, -18, -19) / GLM (Geostationary Lightning Mapper)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=noaa/goes-r-glm). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L2

Level 2 Lightning Cluster-Filter Algorithm (LCFA): events, groups and flashes with location, time, energy and area, in 20-second netCDF files.

Reads one 20 s GLM flash file from the public NOAA bucket on AWS (no credentials) and plots the flash locations coloured by energy.

In [ ]:
%pip install -q s3fs xarray h5netcdf matplotlib numpy

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=True)                # public NOAA bucket, no credentials needed
BUCKET = "noaa-goes19"                               # noaa-goes16, noaa-goes18, noaa-goes19 (GOES-17 until 2023)
PRODUCT = "GLM-L2-LCFA"                             # F = full disk, C = CONUS, M = mesoscale
day = dt.datetime(2025, 6, 1, 12)
prefix = "%s/%s/%s/%s/%s/" % (BUCKET, PRODUCT, day.strftime("%Y"), day.strftime("%j"), day.strftime("%H"))
files = [f for f in fs.ls(prefix) if "_G19_" in f]
print(len(files), "files in", prefix)
if not files:
    raise SystemExit("No files: check the bucket (satellite) and the date")

with fs.open(files[0]) as fobj:
    ds = xr.open_dataset(fobj, engine="h5netcdf").load()
print(ds)

print(ds["flash_lat"].size, "flashes in this 20 s file")
plt.scatter(ds["flash_lon"], ds["flash_lat"], c=ds["flash_energy"], s=8, cmap="plasma")
plt.colorbar(label="flash energy (J)")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.show()